In [1]:
pip install numpy scikit-learn sentence-transformers pandas

In [2]:
import numpy, sklearn, sentence_transformers, pandas
print("numpy", numpy.__version__)
print("scikit-learn", sklearn.__version__)
print("sentence-transformers", sentence_transformers.__version__)
print("pandas", pandas.__version__)

numpy 2.4.6
scikit-learn 1.6.1
sentence-transformers 5.7.0
pandas 2.2.3


In [3]:
%%writefile lab6_search.py
"""Retrieval helper used by every script: TF-IDF or sentence-embedding search."""
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

ENCODER = "sentence-transformers/all-MiniLM-L6-v2"


class Retriever:
    """Index a list of records ({"id": ..., "text": ...}) and return the best matches for a query."""

    def __init__(self, records, backend="tfidf"):
        if not records or any(not rec["text"].strip() for rec in records):
            raise ValueError("Provide nonempty records")
        if backend not in ("tfidf", "embedding"):
            raise ValueError("Use tfidf or embedding")
        self.records = records
        self.backend = backend
        passages = [rec["text"] for rec in records]

        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(passages)
        else:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(ENCODER, device="cpu")
            self.check_length(passages)
            self.matrix = self.model.encode(passages, normalize_embeddings=True)

    def check_length(self, texts):
        """Reject any text longer than the encoder limit (special tokens included)."""
        limit = self.model.max_seq_length
        for text in texts:
            ids = self.model.tokenizer(text, truncation=False)["input_ids"]
            if len(ids) > limit:
                raise ValueError("Input exceeds encoder token limit")

    def _score_all(self, query):
        """Return one score per record, or None when TF-IDF finds no shared word."""
        if self.backend == "tfidf":
            vec = self.vectorizer.transform([query])
            if vec.nnz == 0:
                return None
            return (vec @ self.matrix.T).toarray()[0]
        self.check_length([query])
        vec = self.model.encode([query], normalize_embeddings=True)[0]
        return self.matrix @ vec

    def search(self, query, k=3, filters=None):
        """Top-k (record, score) pairs, highest score first; metadata filters apply first."""
        if not query.strip() or k <= 0:
            return []
        scores = self._score_all(query)
        if scores is None:
            return []
        wanted = filters or {}
        allowed = [i for i, rec in enumerate(self.records)
                   if all(rec.get(key) == value for key, value in wanted.items())]
        allowed.sort(key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in allowed[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Writing lab6_search.py


In [4]:
%%writefile task1_tokens.py
"""Task 1: compare whitespace, word-like (regex) and model sub-word tokens."""
import re

samples = [
    "AI, search!",              # punctuation
    "I can't log in.",          # contraction
    "help@example.com",         # email
    "یہ ایک مثال ہے",            # Urdu
    "",                         # empty string
    "CS4944 version 2.1",       # identifier and number
]


def whitespace_tokens(text):
    return text.split()


def word_like_tokens(text):
    return re.findall(r"\b\w+\b", text, flags=re.UNICODE)


print("=== Whitespace vs word-like tokens ===")
for text in samples:
    plain = whitespace_tokens(text)
    wordy = word_like_tokens(text)
    print(f"Original : {text!r}")
    print(f"  whitespace -> {plain}  ({len(plain)} tokens)")
    print(f"  word-like  -> {wordy}  ({len(wordy)} tokens)")
assert word_like_tokens("") == [] and whitespace_tokens("") == []

print()
print("=== Normalization risk ===")
risky = "I can't log in."
stripped = re.sub(r"[^\w\s]", "", risky.lower())
print(f"Original   : {risky!r}")
print(f"Normalized : {stripped!r}  <- the apostrophe is gone, so 'can't' became 'cant'")

print()
print("=== Model sub-word tokens (all-MiniLM-L6-v2) ===")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
for text in samples[:2]:
    print(f"{text!r} -> {model.tokenizer.tokenize(text)}")

Writing task1_tokens.py


In [5]:
!python task1_tokens.py

=== Whitespace vs word-like tokens ===
Original : 'AI, search!'
  whitespace -> ['AI,', 'search!']  (2 tokens)
  word-like  -> ['AI', 'search']  (2 tokens)
Original : "I can't log in."
  whitespace -> ['I', "can't", 'log', 'in.']  (4 tokens)
  word-like  -> ['I', 'can', 't', 'log', 'in']  (5 tokens)
Original : 'help@example.com'
  whitespace -> ['help@example.com']  (1 tokens)
  word-like  -> ['help', 'example', 'com']  (3 tokens)
Original : 'یہ ایک مثال ہے'
  whitespace -> ['یہ', 'ایک', 'مثال', 'ہے']  (4 tokens)
  word-like  -> ['یہ', 'ایک', 'مثال', 'ہے']  (4 tokens)
Original : ''
  whitespace -> []  (0 tokens)
  word-like  -> []  (0 tokens)
Original : 'CS4944 version 2.1'
  whitespace -> ['CS4944', 'version', '2.1']  (3 tokens)
  word-like  -> ['CS4944', 'version', '2', '1']  (4 tokens)

=== Normalization risk ===
Original   : "I can't log in."
Normalized : 'i cant log in'  <- the apostrophe is gone, so 'can't' became 'cant'

=== Model sub-word tokens (all-MiniLM-L6-v2) ===
modules.j

In [6]:
%%writefile guided_chatbot.py
"""Command-line FAQ chatbot: exact questions get a stored answer, others get suggestions."""
from lab6_search import Retriever

FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?",
     "answer": "Use the recovery link on the login page.",
     "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?",
     "answer": "Use the assignment upload page in the learning portal.",
     "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?",
     "answer": "Request an extension with medical evidence before the deadline.",
     "source": "handbook.txt"},
    # Two FAQs added for Task 1 (synthetic teaching data)
    {"id": "faq:fees", "question": "When is the semester fee due?",
     "answer": "Pay the semester fee at the accounts office before the first day of classes.",
     "source": "fee_rules.txt"},
    {"id": "faq:library", "question": "How many books can I borrow from the library?",
     "answer": "Show your student card at the library desk to borrow up to five books.",
     "source": "library_guide.txt"},
]


def build_bot(backend="tfidf"):
    records = [dict(item, text=item["question"] + " " + item["answer"]) for item in FAQ]
    return Retriever(records, backend)


def reply(engine, query):
    if not query.strip():
        return "Please enter a question."

    # Only an exact canonical question is answered automatically
    asked = query.strip().casefold()
    for item in FAQ:
        if asked == item["question"].casefold():
            return f"{item['answer']} [source: {item['source']}]"

    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."

    lines = ["Possible matches; choose or rephrase your question:"]
    lines += [f"{rec['question']} (similarity {score:.3f})" for rec, score in hits]
    return "\n".join(lines)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf", "embedding"], default="tfidf")
    bot = build_bot(parser.parse_args().backend)

    print("FAQ assistant. Type exit to stop.")
    while True:
        try:
            text = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break
        if text.strip().lower() == "exit":
            break
        print("Bot:", reply(bot, text))

Writing guided_chatbot.py


In [7]:
from guided_chatbot import build_bot, reply

bot = build_bot("tfidf")
conversation = [
    "How do I reset my password?",                      # exact FAQ question
    "password recovery",                                # paraphrase -> suggestions
    "How many books can I borrow from the library?",    # exact (new FAQ)
    "when is fee payment due",                          # paraphrase of a new FAQ
    "",                                                 # empty input
    "volcano geology",                                  # unsupported question
]
for question in conversation:
    print("You:", question)
    print("Bot:", reply(bot, question))
    print()

You: How do I reset my password?
Bot: Use the recovery link on the login page. [source: it_help.txt]

You: password recovery
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.444)
Where do I submit my assignment? (similarity 0.000)
Can illness justify a late submission? (similarity 0.000)

You: How many books can I borrow from the library?
Bot: Show your student card at the library desk to borrow up to five books. [source: library_guide.txt]

You: when is fee payment due
Bot: Possible matches; choose or rephrase your question:
When is the semester fee due? (similarity 0.522)
How do I reset my password? (similarity 0.000)
Where do I submit my assignment? (similarity 0.000)

You: 
Bot: Please enter a question.

You: volcano geology
Bot: No matching FAQ. Please contact the help desk.



In [8]:
from guided_chatbot import build_bot, reply

bot = build_bot("tfidf")
conversation = [
    "How do I reset my password?",                      # exact FAQ question
    "password recovery",                                # paraphrase -> suggestions
    "How many books can I borrow from the library?",    # exact (new FAQ)
    "when is fee payment due",                          # paraphrase of a new FAQ
    "",                                                 # empty input
    "volcano geology",                                  # unsupported question
]
for question in conversation:
    print("You:", question)
    print("Bot:", reply(bot, question))
    print()

You: How do I reset my password?
Bot: Use the recovery link on the login page. [source: it_help.txt]

You: password recovery
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.444)
Where do I submit my assignment? (similarity 0.000)
Can illness justify a late submission? (similarity 0.000)

You: How many books can I borrow from the library?
Bot: Show your student card at the library desk to borrow up to five books. [source: library_guide.txt]

You: when is fee payment due
Bot: Possible matches; choose or rephrase your question:
When is the semester fee due? (similarity 0.522)
How do I reset my password? (similarity 0.000)
Where do I submit my assignment? (similarity 0.000)

You: 
Bot: Please enter a question.

You: volcano geology
Bot: No matching FAQ. Please contact the help desk.



In [9]:
assert "it_help.txt" in reply(bot, "How do I reset my password?")
assert reply(bot, "") == "Please enter a question."
assert "No matching FAQ" in reply(bot, "volcano geology")
assert "Possible matches" in reply(bot, "password recovery")
print("FAQ behaviour checks passed")

FAQ behaviour checks passed


In [10]:
%run guided_chatbot.py

FAQ assistant. Type exit to stop.
You: enter
Bot: No matching FAQ. Please contact the help desk.
You: How do I reset my password
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.624)
Where do I submit my assignment? (similarity 0.178)
How many books can I borrow from the library? (similarity 0.062)
You: exit


In [11]:
!pip install -q numpy scikit-learn sentence-transformers pandas

In [12]:
from google.colab import files

for name in ["lab6_search.py", "task1_tokens.py", "guided_chatbot.py"]:
    files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>